# Notebook 04e — Fine-Tuning AH-PC with Per-Subject-Per-Fold Checkpoints, No Freezing

This notebook is a single-variable ablation of `04d_finetune_per_subject.ipynb`: every
checkpoint, split, and hyperparameter is identical — the only change is that
`BandAttention`/`ChannelAttention` are **no longer frozen**. In `04d` those two modules
had `requires_grad = False`; here all parameters (`band_attn`, `channel_attn`, `encoder`,
`bottleneck`, `projector`, `head`) are trainable from the start.

Motivation: `04d`'s per-subject-pretrained result still trails Sheibani's PC-SSL baseline
(83.02% ± 13.12%). This notebook tests whether letting the attention modules adapt during
fine-tuning (rather than staying frozen at their pre-trained values) closes any more of
that gap.

## Checkpoint location

Checkpoint directory resolution tries the Colab path first, falling back to this
workspace's local path if Colab's Drive isn't mounted (i.e. when running here rather than
in Colab):

1. `/content/drive/MyDrive/AH-PC/models/per_subject/` (Colab)
2. `/workspaces/AH-PC/src/models/per_subject/` (this workspace — where
   `03c_pretrain_per_subject.ipynb` actually saved its checkpoints)

## What this notebook does

1. Re-defines `SheibaniEncoder` + the raw-feature-space projector and `AHPCClassifier`
   (identical to `04d_finetune_per_subject.ipynb`, minus the freezing)
2. Builds the same trial-based 3-fold split as `04_finetune.ipynb`/`04c`/`04d`
3. For each (subject, fold), loads **that pair's own** per-subject-per-fold checkpoint
   and fully fine-tunes a fresh `AHPCClassifier` (25 epochs, same hyperparameters as `04d`)
4. Reports per-subject mean accuracy, overall mean ± std, and compares against the
   PC-SSL baseline, the k=1 globally-pretrained result, and `04d`'s frozen-attention result

## Smoke test

Full training (48 runs × 25 epochs) requires all 48 per-subject-per-fold checkpoints
from `03c_pretrain_per_subject.ipynb` (`RUN_FULL_TRAINING = False` by default). A 2-epoch
smoke test on **Subject 1, Fold 1** runs automatically — that checkpoint already exists
from `03c`'s own smoke test.

In [1]:
import os
import copy
import pickle

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

DATA_DIR = "/workspaces/AH-PC/src/data/processed"

# ── Per-subject checkpoint directory: Colab path first, local workspace fallback ──────
_CKPT_DIR_CANDIDATES = [
    "/content/drive/MyDrive/AH-PC/models/per_subject",   # Colab (Drive mounted)
    "/workspaces/AH-PC/src/models/per_subject",          # this workspace (03c's output)
]

PER_SUBJECT_CKPT_DIR = next((p for p in _CKPT_DIR_CANDIDATES if os.path.isdir(p)), None)
if PER_SUBJECT_CKPT_DIR is None:
    raise FileNotFoundError(
        "No per-subject checkpoint directory found. Checked: "
        + ", ".join(_CKPT_DIR_CANDIDATES)
        + ". Run 03c_pretrain_per_subject.ipynb first."
    )
print(f"Using per-subject checkpoint directory: {PER_SUBJECT_CKPT_DIR}")

FINETUNE_CKPT_DIR = "/workspaces/AH-PC/src/models/finetune_no_freeze"

EMOTIONS = ["Disgust", "Fear", "Sad", "Neutral", "Happy"]
NUM_SUBJECTS = 16


def get_subject_fold_ckpt_path(subject_id, fold_idx):
    """Path to the pre-trained checkpoint specific to this (subject, fold) pair,
    as produced by 03c_pretrain_per_subject.ipynb."""
    return os.path.join(
        PER_SUBJECT_CKPT_DIR, f"ahpc_subject{subject_id:02d}_fold{fold_idx + 1}_best.pt"
    )

PyTorch version: 2.12.1+cu130
Device: cpu
Using per-subject checkpoint directory: /workspaces/AH-PC/src/models/per_subject


## 1. Re-defining the AH-PC Architecture

Identical to `04c_finetune_k1_raw.ipynb`: only `SheibaniEncoder` and the current
raw-feature-space projector (`Flatten(4800) → Linear(4800,256) + LayerNorm`) are needed —
`AHPCClassifier` never uses a decoder head regardless of which per-subject checkpoint it
loads.

In [2]:
class BandAttention(nn.Module):
    def __init__(self, num_bands):
        super().__init__()
        self.fc1 = nn.Linear(num_bands * 2, num_bands // 2)
        self.fc2 = nn.Linear(num_bands // 2, num_bands)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=1)
        max_pool = x.max(dim=1).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(1)


class ChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.fc1 = nn.Linear(num_channels * 2, num_channels // 2)
        self.fc2 = nn.Linear(num_channels // 2, num_channels)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=2)
        max_pool = x.max(dim=2).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(2)


class SheibaniEncoder(nn.Module):
    """Sheibani's attention + CNN encoder, reproduced exactly (Notebooks 02/03/03b/03c)."""

    OUT_CHANNELS = 64

    def __init__(self):
        super().__init__()
        self.band_attn = BandAttention(num_bands=5)
        self.channel_attn = ChannelAttention(num_channels=62)

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

    def forward(self, x):  # x: (B, 62, 5)
        x = self.band_attn(x)
        x = self.channel_attn(x)
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.bottleneck(x)
        return x  # (B, 64, 15, 5)


LATENT_DIM      = 256
BOTTLENECK_FLAT = 64 * 15 * 5  # 4800 — flattened (C, H, W) bottleneck

print("SheibaniEncoder re-defined (matches Notebooks 03/03b/03c). "
      "AHPCModel itself is not needed here — AHPCClassifier only restores "
      "sheibani_encoder.* and projector.* tensors.")

SheibaniEncoder re-defined (matches Notebooks 03/03b/03c). AHPCModel itself is not needed here — AHPCClassifier only restores sheibani_encoder.* and projector.* tensors.


## 2. Loading a Pre-Trained Encoder — No Freezing

Same as `04d_finetune_per_subject.ipynb`, but `BandAttention`/`ChannelAttention` are
**not** frozen — every parameter is trainable from the start.

- **Trainable:** everything — `band_attn`, `channel_attn`, the CNN (`encoder`), the
  `bottleneck`, the `projector`, and the new head
- **Head:** `Linear(256→128) → ReLU → Dropout(0.3) → Linear(128→5)`

In [3]:
class AHPCClassifier(nn.Module):
    """
    AH-PC encoder + projector with no freezing: every parameter (BandAttention,
    ChannelAttention, CNN, bottleneck, projector, head) is trainable.
    `pretrained_ckpt_path` may point at any per-subject-per-fold checkpoint from
    03c_pretrain_per_subject.ipynb.
    """

    def __init__(self, pretrained_ckpt_path, num_classes=5, latent_dim=LATENT_DIM):
        super().__init__()

        self.encoder = SheibaniEncoder()
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(BOTTLENECK_FLAT, latent_dim),
            nn.LayerNorm(latent_dim),
        )
        self._load_pretrained(pretrained_ckpt_path)

        self.head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def _load_pretrained(self, path):
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing per-subject pretrained checkpoint: {path}. "
                "Run 03c_pretrain_per_subject.ipynb (with RUN_FULL_TRAINING=True for "
                "the full protocol) to produce it first."
            )
        # weights_only=False: this is our own checkpoint (Notebook 03c), which stores
        # plain Python/numpy history alongside the tensors, not an untrusted file.
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        full_sd = ckpt["model_state_dict"]

        enc_sd = {
            k[len("sheibani_encoder."):]: v
            for k, v in full_sd.items() if k.startswith("sheibani_encoder.")
        }
        proj_sd = {
            k[len("projector."):]: v
            for k, v in full_sd.items() if k.startswith("projector.")
        }
        self.encoder.load_state_dict(enc_sd)
        self.projector.load_state_dict(proj_sd)

    def get_embedding(self, x):
        feat = self.encoder(x)
        return self.projector(feat)

    def forward(self, x):
        z = self.get_embedding(x)
        return self.head(z)


_ckpt_path = get_subject_fold_ckpt_path(1, 0)
_model = AHPCClassifier(_ckpt_path)
_trainable_params = [p for p in _model.parameters() if p.requires_grad]
_total_params = list(_model.parameters())
n_trainable = sum(p.numel() for p in _trainable_params)
n_total = sum(p.numel() for p in _total_params)
print(f"Loaded from: {_ckpt_path}")
print(f"Trainable params (all layers, no freezing): {n_trainable:,} / {n_total:,} total")
del _ckpt_path, _model, _trainable_params, _total_params

Loaded from: /workspaces/AH-PC/src/models/per_subject/ahpc_subject01_fold1_best.pt
Trainable params (all layers, no freezing): 1,292,525 / 1,292,525 total


## 3. Dataset: Subject-Dependent 3-Fold CV, Split by Trial

Identical to `04_finetune.ipynb`/`04c_finetune_k1_raw.ipynb` — each subject's 45 trials
are 3 sessions of 15 trials each, laid out contiguously as
`[session 1: 0-14][session 2: 15-29][session 3: 30-44]`. Each fold holds out one
session's trials as the test set and trains on the remaining two:

- **Fold 1:** train = sessions 2+3 (trials 15-44), test = session 1 (trials 0-14)
- **Fold 2:** train = sessions 1+3 (trials 0-14, 30-44), test = session 2 (trials 15-29)
- **Fold 3:** train = sessions 1+2 (trials 0-29), test = session 3 (trials 30-44)

In [4]:
class EmotionDataset(Dataset):
    def __init__(self, pairs):
        self.samples = pairs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p = self.samples[idx]
        anchor = torch.tensor(p["anchor"], dtype=torch.float32)  # (62, 5)
        label = torch.tensor(p["label"], dtype=torch.long)
        return anchor, label


SESSION_TRIALS = [list(range(0, 15)), list(range(15, 30)), list(range(30, 45))]


def get_fold_trials(fold_idx: int):
    """fold_idx in {0, 1, 2}. Returns (train_trials, test_trials) for that fold,
    holding out one session's trials as test and training on the other two."""
    test_trials = SESSION_TRIALS[fold_idx]
    train_trials = [t for i, s in enumerate(SESSION_TRIALS) if i != fold_idx for t in s]
    return sorted(train_trials), sorted(test_trials)


def load_subject_pairs(subject_id: int, data_dir=DATA_DIR):
    path = os.path.join(data_dir, f"subject_{subject_id:02d}_pairs.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_subject_fold_pairs(subject_id: int, fold_idx: int):
    """Split one subject's pairs into train/test by trial index for the given fold."""
    pairs = load_subject_pairs(subject_id)
    train_trials, test_trials = get_fold_trials(fold_idx)
    train_set, test_set = set(train_trials), set(test_trials)

    train_pairs = [p for p in pairs if p["trial"] in train_set]
    test_pairs = [p for p in pairs if p["trial"] in test_set]
    return train_pairs, test_pairs, train_trials, test_trials


print("Trial-based fold splitting defined.")
for fold_idx in range(3):
    train_trials, test_trials = get_fold_trials(fold_idx)
    print(f"  Fold {fold_idx + 1}: train trials={train_trials[0]}-{train_trials[-1]} "
          f"(n={len(train_trials)}), test trials={test_trials[0]}-{test_trials[-1]} "
          f"(n={len(test_trials)})")

Trial-based fold splitting defined.
  Fold 1: train trials=15-44 (n=30), test trials=0-14 (n=15)
  Fold 2: train trials=0-44 (n=30), test trials=15-29 (n=15)
  Fold 3: train trials=0-29 (n=30), test trials=30-44 (n=15)


## 4. Class Imbalance and Inverse-Frequency Weights

Identical to `04_finetune.ipynb`/`04c` — inverse-frequency weights computed from each
fold's own training labels.

In [5]:
def compute_class_weights(labels, num_classes=5):
    """Inverse-frequency weights from a fold's own training labels.
    Classes absent from a small per-subject fold are clipped to count=1 to avoid div-by-zero."""
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.clip(counts, 1, None)
    weights = counts.max() / counts
    return torch.tensor(weights, dtype=torch.float32)


_train_pairs, _test_pairs, _, _ = get_subject_fold_pairs(1, 0)
_train_labels = np.array([p["label"] for p in _train_pairs])
_weights = compute_class_weights(_train_labels)

print("Example — Subject 1, Fold 1 training-fold class weights:")
for name, count, weight in zip(EMOTIONS, np.bincount(_train_labels, minlength=5), _weights.tolist()):
    print(f"  {name:10s}  count={int(count):5d}  weight={weight:.3f}")
del _train_pairs, _test_pairs, _train_labels, _weights

Example — Subject 1, Fold 1 training-fold class weights:
  Disgust     count=  149  weight=1.718
  Fear        count=  184  weight=1.391
  Sad         count=  256  weight=1.000
  Neutral     count=  185  weight=1.384
  Happy       count=  128  weight=2.000


## 5. Training: One Model Per Subject Per Fold, From Its Own Checkpoint

Same hyperparameters as `04d_finetune_per_subject.ipynb`:

- **Optimizer:** AdamW, `lr=8e-5`, `weight_decay=1e-4`
- **Scheduler:** `StepLR(step_size=10, gamma=0.5)`
- **Loss:** `CrossEntropyLoss` with that fold's inverse-frequency class weights
- **Epochs:** 25 · **Batch size:** 32

The only structural difference from `04d`: no parameters are frozen, so
`trainable_params` here is every parameter in the model. `AHPCClassifier` is still
initialised from `get_subject_fold_ckpt_path(subject_id, fold_idx)` — that exact
subject/fold's own per-subject pre-trained encoder. Best-test-accuracy weights save to
`src/models/finetune_no_freeze/subject{sid:02d}_fold{fold}_best.pt`.

In [6]:
def train_subject_fold(subject_id, fold_idx, epochs=25, batch_size=32, lr=8e-5,
                        weight_decay=1e-4, step_size=10, gamma=0.5,
                        device=DEVICE, verbose=True):
    train_pairs, test_pairs, train_trials, test_trials = get_subject_fold_pairs(subject_id, fold_idx)

    train_ds = EmotionDataset(train_pairs)
    test_ds = EmotionDataset(test_pairs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    train_labels = np.array([p["label"] for p in train_pairs])
    class_weights = compute_class_weights(train_labels).to(device)

    ckpt_path = get_subject_fold_ckpt_path(subject_id, fold_idx)
    model = AHPCClassifier(ckpt_path).to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_acc = -1.0
    best_state = None
    best_preds, best_labels = None, None
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for anchor, label in train_loader:
            anchor, label = anchor.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(anchor)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * anchor.size(0)
        scheduler.step()
        avg_loss = running_loss / len(train_ds)

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for anchor, label in test_loader:
                logits = model(anchor.to(device))
                preds.extend(logits.argmax(dim=1).cpu().numpy())
                labels.extend(label.numpy())
        acc = accuracy_score(labels, preds)

        history["train_loss"].append(avg_loss)
        history["test_acc"].append(acc)

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
            best_preds, best_labels = preds, labels

        if verbose:
            print(f"    Epoch {epoch:2d}/{epochs}  loss={avg_loss:.4f}  test_acc={acc:.4f}")

    os.makedirs(FINETUNE_CKPT_DIR, exist_ok=True)
    out_path = os.path.join(FINETUNE_CKPT_DIR, f"subject{subject_id:02d}_fold{fold_idx + 1}_best.pt")
    torch.save({"model_state_dict": best_state, "best_acc": best_acc}, out_path)

    return {
        "subject": subject_id,
        "fold": fold_idx + 1,
        "accuracy": best_acc,
        "y_true": np.array(best_labels),
        "y_pred": np.array(best_preds),
        "history": history,
        "ckpt_path": out_path,
        "pretrained_ckpt_path": ckpt_path,
        "n_train": len(train_ds),
        "n_test": len(test_ds),
    }


print("train_subject_fold() defined.")

train_subject_fold() defined.


## 6. Full Experiment: All 16 Subjects × 3 Folds

Requires all 48 per-subject-per-fold checkpoints from `03c_pretrain_per_subject.ipynb`
(`RUN_FULL_TRAINING = True` there). **Intentionally left off**
(`RUN_FULL_TRAINING = False`) until the smoke test below has been validated.

In [7]:
RUN_FULL_TRAINING = False  # set True to launch all 16 subjects x 3 folds (48 runs x 25 epochs)

fold_results = []  # one dict per (subject, fold), from train_subject_fold()

if RUN_FULL_TRAINING:
    for subject_id in range(1, NUM_SUBJECTS + 1):
        print(f"\nSubject {subject_id}")
        for fold_idx in range(3):
            print(f"  Fold {fold_idx + 1}/3")
            res = train_subject_fold(subject_id, fold_idx, epochs=25, batch_size=32, verbose=False)
            fold_results.append(res)
            print(f"    -> best test_acc = {res['accuracy']:.4f}")
    print("\nTraining complete for all subjects and folds.")
else:
    print("RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. "
          "See Section 9 for a smoke test of the pipeline instead.")

RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. See Section 9 for a smoke test of the pipeline instead.


## 7. Evaluation

Once `fold_results` is populated (after a full run):

- **Per-subject mean accuracy** across its 3 folds (± std)
- **Overall mean ± std across all 16 subjects**
- An **aggregated classification report** and **normalized confusion matrix**

In [8]:
from collections import defaultdict


def plot_confusion_matrix(cm, class_names=EMOTIONS, title="Confusion Matrix (Normalized)"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    plt.show()


overall_mean = overall_std = None

if not fold_results:
    print("No results yet — set RUN_FULL_TRAINING=True in Section 6 and re-run to "
          "produce evaluation metrics.")
else:
    acc_by_subject = defaultdict(list)
    for r in fold_results:
        acc_by_subject[r["subject"]].append(r["accuracy"])

    print("Per-subject mean accuracy across 3 folds:")
    subject_means = []
    for sid in sorted(acc_by_subject):
        accs = acc_by_subject[sid]
        m, s = np.mean(accs), np.std(accs)
        subject_means.append(m)
        fold_str = ", ".join(f"{a * 100:.2f}%" for a in accs)
        print(f"  Subject {sid:2d}: {m * 100:5.2f}% ± {s * 100:5.2f}%  (folds: {fold_str})")

    overall_mean = float(np.mean(subject_means) * 100)
    overall_std = float(np.std(subject_means) * 100)
    print(f"\nOverall mean accuracy across all {len(subject_means)} subjects: "
          f"{overall_mean:.2f}% ± {overall_std:.2f}%")

    all_y_true = np.concatenate([r["y_true"] for r in fold_results])
    all_y_pred = np.concatenate([r["y_pred"] for r in fold_results])

    print("\nAggregated classification report (all subjects, all folds):")
    print(classification_report(all_y_true, all_y_pred, target_names=EMOTIONS, digits=3))

    agg_cm = confusion_matrix(all_y_true, all_y_pred, labels=[0, 1, 2, 3, 4])
    plot_confusion_matrix(agg_cm, title="Aggregated Confusion Matrix (Normalized) — Per-Subject Pretrain, No Freeze")

No results yet — set RUN_FULL_TRAINING=True in Section 6 and re-run to produce evaluation metrics.


## 8. Comparison Against Baselines

Sheibani's PC-SSL baseline: **83.02% ± 13.12%**. The k=1 globally-pretrained AH-PC
variant (`04c_finetune_k1_raw.ipynb`, one shared encoder pre-trained across subjects
1-13): **58.53% ± 12.46%**. This section adds the no-freeze, per-subject-pretrained
result for comparison against both baselines and against `04d`'s frozen-attention
per-subject result — if unfreezing `band_attn`/`channel_attn` closes more of the gap to
the PC-SSL baseline, that's evidence the frozen attention modules (not the raw-feature
decoding scheme) were the remaining bottleneck.

In [9]:
PC_SSL_BASELINE_MEAN = 83.02
PC_SSL_BASELINE_STD = 13.12

AH_PC_GLOBAL_K1_MEAN = 58.53
AH_PC_GLOBAL_K1_STD = 12.46

print(f"{'PC-SSL baseline:':30s} {PC_SSL_BASELINE_MEAN:.2f}% ± {PC_SSL_BASELINE_STD:.2f}%")
print(f"{'AH-PC k=1 global pretrain:':30s} {AH_PC_GLOBAL_K1_MEAN:.2f}% ± {AH_PC_GLOBAL_K1_STD:.2f}%")

if overall_mean is None:
    print(f"{'AH-PC per-subject, no freeze:':30s} XX.XX% ± XX.XX%")
    print("\nRun the full 16-subject x 3-fold protocol (Section 6) to fill this in — "
          "requires all 48 checkpoints from 03c_pretrain_per_subject.ipynb.")
else:
    print(f"{'AH-PC per-subject, no freeze:':30s} {overall_mean:.2f}% ± {overall_std:.2f}%")
    print(f"\n{'vs. PC-SSL baseline:':30s} {overall_mean - PC_SSL_BASELINE_MEAN:+.2f} pts")
    print(f"{'vs. AH-PC k=1 global pretrain:':30s} {overall_mean - AH_PC_GLOBAL_K1_MEAN:+.2f} pts")

PC-SSL baseline:               83.02% ± 13.12%
AH-PC k=1 global pretrain:     58.53% ± 12.46%
AH-PC per-subject, no freeze:  XX.XX% ± XX.XX%

Run the full 16-subject x 3-fold protocol (Section 6) to fill this in — requires all 48 checkpoints from 03c_pretrain_per_subject.ipynb.


## 9. Smoke Test

2 epochs on **Subject 1, Fold 1** using its own per-subject checkpoint
(`ahpc_subject01_fold1_best.pt`, produced by `03c_pretrain_per_subject.ipynb`'s own
smoke test — no full 48-run protocol required for this check). Validates:

1. **The per-subject checkpoint resolves and loads correctly** for this (subject, fold).
2. **The trial-based split is correct.**
3. **Every module changes during training**, including `BandAttention`/`ChannelAttention`
   — confirming no freezing is in effect (unlike `04d`, where those two stayed frozen).
4. **Predictions come out in a sane shape.**

In [10]:
print("=" * 70)
print("SMOKE TEST: 2 epochs on Subject 1, Fold 1 (per-subject checkpoint, no freeze)")
print("=" * 70)

SMOKE_SUBJECT = 1
SMOKE_FOLD = 0  # Fold 1 (0-based)
SMOKE_EPOCHS = 2

smoke_ckpt_path = get_subject_fold_ckpt_path(SMOKE_SUBJECT, SMOKE_FOLD)
print(f"Per-subject checkpoint: {smoke_ckpt_path}\n")

train_pairs, test_pairs, train_trials, test_trials = get_subject_fold_pairs(SMOKE_SUBJECT, SMOKE_FOLD)

print(f"Train trials ({len(train_trials)}): {train_trials}")
print(f"Test trials  ({len(test_trials)}): {test_trials}")
print(f"Train pairs: {len(train_pairs):,}")
print(f"Test pairs:  {len(test_pairs):,}\n")

train_ds = EmotionDataset(train_pairs)
test_ds = EmotionDataset(test_pairs)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

train_labels = np.array([p["label"] for p in train_pairs])
class_weights = compute_class_weights(train_labels).to(DEVICE)

model = AHPCClassifier(smoke_ckpt_path).to(DEVICE)

params_before = {
    "band_attn": {k: v.clone() for k, v in model.encoder.band_attn.state_dict().items()},
    "channel_attn": {k: v.clone() for k, v in model.encoder.channel_attn.state_dict().items()},
    "cnn": {k: v.clone() for k, v in model.encoder.encoder.state_dict().items()},
    "bottleneck": {k: v.clone() for k, v in model.encoder.bottleneck.state_dict().items()},
    "projector": {k: v.clone() for k, v in model.projector.state_dict().items()},
    "head": {k: v.clone() for k, v in model.head.state_dict().items()},
}

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=8e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(1, SMOKE_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for anchor, label in train_loader:
        anchor, label = anchor.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(anchor)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * anchor.size(0)
    scheduler.step()
    print(f"  Epoch {epoch}/{SMOKE_EPOCHS}  train_loss={running_loss / len(train_ds):.4f}")

params_after = {
    "band_attn": {k: v.clone() for k, v in model.encoder.band_attn.state_dict().items()},
    "channel_attn": {k: v.clone() for k, v in model.encoder.channel_attn.state_dict().items()},
    "cnn": {k: v.clone() for k, v in model.encoder.encoder.state_dict().items()},
    "bottleneck": {k: v.clone() for k, v in model.encoder.bottleneck.state_dict().items()},
    "projector": {k: v.clone() for k, v in model.projector.state_dict().items()},
    "head": {k: v.clone() for k, v in model.head.state_dict().items()},
}

print()
all_changed = True
for name, before in params_before.items():
    after = params_after[name]
    changed = any(not torch.equal(before[k], after[k]) for k in before)
    all_changed &= changed
    print(f"{name:14s} changed after training:   {changed}")

assert all_changed, ("FAIL: a module did not change during training — with no freezing, "
                      "every module (including band_attn/channel_attn) should update.")
print("\nAll smoke-test assertions passed (no module is frozen).")

SMOKE TEST: 2 epochs on Subject 1, Fold 1 (per-subject checkpoint, no freeze)
Per-subject checkpoint: /workspaces/AH-PC/src/models/per_subject/ahpc_subject01_fold1_best.pt

Train trials (30): [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Test trials  (15): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Train pairs: 902
Test pairs:  561



  Epoch 1/2  train_loss=1.5151


  Epoch 2/2  train_loss=1.0190

band_attn      changed after training:   True
channel_attn   changed after training:   True
cnn            changed after training:   True
bottleneck     changed after training:   True
projector      changed after training:   True
head           changed after training:   True

All smoke-test assertions passed (no module is frozen).


In [11]:
model.eval()
anchor_sample, label_sample = next(iter(test_loader))
with torch.no_grad():
    logits = model(anchor_sample.to(DEVICE))
    probs = F.softmax(logits, dim=-1)
    preds = probs.argmax(dim=1).cpu()

print("Sample predictions vs actual labels (2 epochs only — expect near-chance):\n")
n_show = min(10, len(preds))
for i in range(n_show):
    pred_name = EMOTIONS[preds[i].item()]
    true_name = EMOTIONS[label_sample[i].item()]
    match = "✓" if pred_name == true_name else " "
    print(f"  [{match}] Predicted: {pred_name:10s}  Actual: {true_name:10s}")

smoke_acc = accuracy_score(label_sample.numpy(), preds.numpy())
print(f"\nSmoke-test batch accuracy: {smoke_acc:.3f} (n={len(preds)}, chance level = 0.20)")

Sample predictions vs actual labels (2 epochs only — expect near-chance):

  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     
  [✓] Predicted: Happy       Actual: Happy     

Smoke-test batch accuracy: 0.500 (n=32, chance level = 0.20)


---
## Summary

| Step | Status |
|------|--------|
| Checkpoint dir resolved (Colab / local fallback) | depends on run |
| Architecture re-defined (raw-feature-space projector, matches 03c) | depends on run |
| Trial-based 3-fold split (identical convention to 04/04c) | depends on run |
| Per-(subject, fold) fine-tuning from its own pre-trained checkpoint | depends on run |
| Smoke test: Subject 1, Fold 1, 2 epochs | depends on run |
| Full protocol: 48 fine-tuned classifiers | not launched by default |
| Three-way comparison vs. PC-SSL baseline and k=1 global pretrain | depends on run |

**Next step:** run `03c_pretrain_per_subject.ipynb` with `RUN_FULL_TRAINING = True` to
produce all 48 per-subject checkpoints, then set `RUN_FULL_TRAINING = True` here to fill
in the per-subject row of the comparison table.